# Frozen Qwen3-Embedding-4B, 2,048 tokens

**Question (Further Work item 4):** does reading more of each message help? Same model, data, split and classifier settings as `Frozen Qwen3-Embedding-4B.ipynb`, but each message is cut at 2,048 tokens instead of 512.

About 10% of the posts are longer than 512 tokens, and they hold 59% of all the text. At 2,048 tokens, 83% of them are read in full (`Lessons.md`, "GPU memory and context length"). But long posts aren't where the errors are. In 5-fold CV on train (LR on the 512-token vectors), posts over 512 tokens were misclassified 10% of the time, vs. 20% for shorter ones, and 41% of posts under 64 tokens were wrong. A longer window can't help those short posts.

**Predict before running:** at 512 tokens this model got 77.63 (LR) and 77.54 (MLP) on test. How much will 2,048 tokens add?

## Load dataset (copied from the PLAYGROUND)

In [1]:
import numpy as np
import torch
import torch.nn as nn
from sklearn.datasets import fetch_20newsgroups

newsgroups_train = fetch_20newsgroups(
    subset='train', remove=('headers', 'footers', 'quotes'))

newsgroups_test = fetch_20newsgroups(
    subset='test', remove=('headers', 'footers', 'quotes'))

# Remove empty documents
for s, name in zip([newsgroups_train, newsgroups_test], ["train", "test"]):
  empty_indices = {i for i, doc in enumerate(s.data) if len(doc) == 0}
  orig_len = len(s.data)
  for k in ['data', 'filenames', 'target']:
    s[k] = [s[k][i] for i in range(orig_len) if i not in empty_indices]
  print(f"Removed {len(empty_indices)} empty documents from the {name} set. Before: {orig_len}. After: {len(s.data)}.")

y_train = np.array(newsgroups_train.target)
y_test = np.array(newsgroups_test.target)
n_classes = len(newsgroups_train.target_names)

Removed 218 empty documents from the train set. Before: 11314. After: 11096.
Removed 162 empty documents from the test set. Before: 7532. After: 7370.


## Qwen3-Embedding-4B last-token embeddings, up to 2,048 tokens

Same model and instruction as the 512-token notebook. Two changes:
- **`MAX_LENGTH = 2048`.**
- **Batches sorted by length, with a token budget.** A batch is padded to its longest message, and in random order almost every batch contains a long post. At 2,048 tokens most of the computation would go on padding. Instead, the messages go through shortest first, and each batch holds as many as fit in 8,192 tokens, padding included (the old batch: 16 × 512). Peak GPU memory depends only on the tokens per batch, so it stays at about 9.3 GB. (Checked: batches of 16 × 512, 8 × 1,024 and 4 × 2,048 all peaked at 9.31–9.32 GB.) The vectors are put back in the original order.

The result is saved to `qwen3_4b_2048_last_20ng.npz` (about 10 minutes on the GPU: half the 512-token notebook's time, thanks to the sorting).

In [2]:
import os
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModel

device = "cuda" if torch.cuda.is_available() else "cpu"
CACHE = "qwen3_4b_2048_last_20ng.npz"
INSTRUCTION = "Instruct: Identify the topic or theme of the given news articles\nQuery:"
MAX_LENGTH = 2048
TOKENS_PER_BATCH = 8192  # as many tokens per batch as 16 x 512 in the 512-token notebook: the same peak GPU memory

## X is a list of strings (messages). Returns a np 2D array of shape (len(X), 2560): the last-token embedding of
## each message (instruction in front, up to MAX_LENGTH tokens), in the original order.
## Messages go through shortest first, in batches of up to TOKENS_PER_BATCH tokens (padding included), so one long
## message never makes a whole batch of short ones pad up to its length.
def qwen_embed(X, tokenizer, qwen):
    texts = [INSTRUCTION + x for x in X]
    lengths = [len(ids) for ids in tokenizer(texts, truncation=True, max_length=MAX_LENGTH)["input_ids"]]
    order = np.argsort(lengths, kind="stable")
    batches, i = [], 0
    while i < len(order):
        j = i + 1  # grow the batch while (messages in it) x (its longest message) still fits in the budget
        while j < len(order) and (j - i + 1) * lengths[order[j]] <= TOKENS_PER_BATCH:
            j += 1
        batches.append(order[i:j])
        i = j

    vecs = np.empty((len(X), qwen.config.hidden_size), dtype=np.float32)
    for idx in tqdm(batches):
        batch = tokenizer([texts[k] for k in idx], padding=True, truncation=True, max_length=MAX_LENGTH,
                          return_tensors="pt").to(device)
        with torch.no_grad():  # frozen: no gradients needed
            out = qwen(**batch)
        # Padding is on the left, so position -1 is every message's last token (<|endoftext|>)
        vecs[idx] = out.last_hidden_state[:, -1].float().cpu().numpy()
    return vecs

if os.path.exists(CACHE):
    saved = np.load(CACHE)
    X_train, X_test = saved["X_train"], saved["X_test"]
else:
    # padding_side="left" so that every message's last token ends up at the same position, -1
    tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-Embedding-4B", padding_side="left")
    # bfloat16 is the model's native precision and halves its memory
    qwen = AutoModel.from_pretrained("Qwen/Qwen3-Embedding-4B", torch_dtype=torch.bfloat16).to(device).eval()
    X_train = qwen_embed(newsgroups_train.data, tokenizer, qwen)
    X_test = qwen_embed(newsgroups_test.data, tokenizer, qwen)
    np.savez(CACHE, X_train=X_train, X_test=X_test)

print(X_train.shape, X_test.shape)

/home/tonyxdsu/venvs/cs423/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


(11096, 2560) (7370, 2560)


## Baseline: logistic regression

First, hold out 10% of train as a dev set, the same split as the RoBERTa notebooks. The MLP needs it for early stopping; the test set is only used at the very end. LR trains on the same 90%, so the comparison stays fair.

LR uses the RoBERTa search's best settings:
- **Standardized inputs** (mean 0 and std 1 in every dimension, from the 90%'s statistics only). Qwen 4B's values spread about 77× wider than RoBERTa's `<s>` values (median per-dimension std 1.95 vs. 0.025). On raw inputs, the same `C` would mean a different amount of regularization. Standardized, both encoders are on the same scale, so RoBERTa's best `C` is a fair guess.
- **`C = 0.003`**, the best for standardized RoBERTa features (70.78 CV, 66.77 test).
- **1 BLAS thread** (`threadpool_limits(1)`), which is about 60× faster on this machine.

In [3]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn import metrics
from threadpoolctl import threadpool_limits

X_tr, X_dev, y_tr, y_dev = train_test_split(
    X_train, y_train, test_size=0.1, stratify=y_train, random_state=423)

scaler = StandardScaler().fit(X_tr)  # mean and std of the 90% only, so dev and test stay unseen
X_tr, X_dev = scaler.transform(X_tr), scaler.transform(X_dev)

with threadpool_limits(1):
    logreg = LogisticRegression(C=0.003, max_iter=1000).fit(X_tr, y_tr)
lr_acc = metrics.accuracy_score(y_test, logreg.predict(scaler.transform(X_test)))
print(f"LR test accuracy: {lr_acc*100:.2f}")

LR test accuracy: 77.58


## MLP

2560 → 2048 → 64 → 20, the best RoBERTa config from the search (with Qwen 4B's 2,560-d input instead of 768). The search found that these mattered:
- **A small learning rate:** AdamW at lr 1e-4. The first notebook's 1e-3 is too big a step once inputs are standardized. Weight decay stays at AdamW's default, 0.01.
- **Heavy dropout in every place:** 0.2 on the input, 0.7 after hidden layer 1, 0.5 after hidden layer 2. Without dropout, the MLP was only level with LR.
- **Label smoothing 0.1:** the training target puts 90% of the probability on the right class and spreads the other 10% evenly over all classes, so the model is never pushed toward 100% confidence.

GELU and batch size 32 also come from the search, but barely mattered there.

In [4]:
torch.manual_seed(423)

mlp = nn.Sequential(
    nn.Dropout(0.2),                                    # dropout on the input embedding
    nn.Linear(2560, 2048), nn.GELU(), nn.Dropout(0.7),  # hidden layer 1: wide, heavy dropout
    nn.Linear(2048, 64), nn.GELU(), nn.Dropout(0.5),    # hidden layer 2: narrow
    nn.Linear(64, n_classes),                           # one logit per class
).to(device)

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = torch.optim.AdamW(mlp.parameters(), lr=1e-4)  # weight decay: AdamW's default, 0.01

Training loop with early stopping, as in the A1 search: stop once dev loss hasn't improved for `PATIENCE` epochs, then go back to the best epoch's weights. Two changes from the first notebook, both from the search:
- **Patience 10 and up to 300 epochs** (was 5 and 100). A 10× smaller learning rate needs more epochs: at lr 1e-4, RoBERTa's best epoch was around 50–90.
- **Dev loss is plain cross-entropy, without label smoothing.** The train loss includes smoothing, so it can't go below about 0.6 and isn't directly comparable to the dev loss.

In [5]:
import copy
import time
from torch.utils.data import TensorDataset, DataLoader

loader = DataLoader(TensorDataset(torch.tensor(X_tr), torch.tensor(y_tr)), batch_size=32, shuffle=True)
X_dev_t, y_dev_t = torch.tensor(X_dev).to(device), torch.tensor(y_dev).to(device)

PATIENCE = 10
best = {"dev_loss": float("inf"), "epoch": 0, "state": None}
start = time.perf_counter()

for epoch in range(1, 301):
    mlp.train()  # dropout on
    train_loss = 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        loss = criterion(mlp(xb), yb)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * len(xb)

    mlp.eval()  # dropout off
    with torch.no_grad():
        dev_logits = mlp(X_dev_t)
        dev_loss = nn.functional.cross_entropy(dev_logits, y_dev_t).item()  # no label smoothing, as in the search
        dev_acc = (dev_logits.argmax(dim=1) == y_dev_t).float().mean().item()
    print(f"epoch {epoch:3d}  train loss {train_loss / len(X_tr):.3f}  dev loss {dev_loss:.3f}  dev acc {dev_acc*100:.2f}")

    if dev_loss < best["dev_loss"]:
        best = {"dev_loss": dev_loss, "epoch": epoch, "state": copy.deepcopy(mlp.state_dict())}
    elif epoch - best["epoch"] >= PATIENCE:
        break

mlp.load_state_dict(best["state"])
print(f"Training took {time.perf_counter() - start:.1f} s. Using epoch {best['epoch']} (lowest dev loss)")

epoch   1  train loss 1.640  dev loss 0.748  dev acc 79.82


epoch   2  train loss 1.363  dev loss 0.700  dev acc 80.54


epoch   3  train loss 1.299  dev loss 0.668  dev acc 81.08


epoch   4  train loss 1.266  dev loss 0.645  dev acc 81.89


epoch   5  train loss 1.241  dev loss 0.632  dev acc 81.53


epoch   6  train loss 1.223  dev loss 0.631  dev acc 81.89


... (18 outputs left out)

epoch  25  train loss 1.026  dev loss 0.593  dev acc 82.34


epoch  26  train loss 1.023  dev loss 0.595  dev acc 82.43


epoch  27  train loss 1.020  dev loss 0.582  dev acc 83.33


epoch  28  train loss 1.011  dev loss 0.580  dev acc 82.88
Training took 78.6 s. Using epoch 18 (lowest dev loss)


## LR vs. MLP on the test set

In [6]:
mlp.eval()
with torch.no_grad():
    mlp_pred = mlp(torch.tensor(scaler.transform(X_test)).to(device)).argmax(dim=1).cpu().numpy()
mlp_acc = metrics.accuracy_score(y_test, mlp_pred)

print(f"LR  test accuracy: {lr_acc*100:.2f}")
print(f"MLP test accuracy: {mlp_acc*100:.2f}")
print(f"Difference: {(mlp_acc - lr_acc)*100:+.2f} points (1 point = {len(y_test) / 100:.0f} test messages)")

LR  test accuracy: 77.58
MLP test accuracy: 77.71
Difference: +0.12 points (1 point = 74 test messages)


For comparison (test accuracy, same classifier settings):

| | LR | MLP |
|---|---|---|
| Qwen3-Embedding-4B, 512 tokens (`Frozen Qwen3-Embedding-4B.ipynb`) | 77.63 | 77.54 |
| Qwen3-Embedding-0.6B, 512 tokens (`Frozen Qwen3-Embedding-0.6B.ipynb`) | 72.90 | 74.15 |
| RoBERTa `<s>`, tuned (`hparam_search/final_test.json`; MLP: mean of 10 seeds) | 66.77 | 68.33 |

This is one MLP with one seed, with settings tuned for RoBERTa, so read a gap under about a point as "about the same". The longer window only changes the vectors of the ~10% of messages over 512 tokens, and most of those were already right, so any real gain here is small.

# Tuned classifiers

Added after tuning both classifiers on these 2,048-token vectors: LR by a `C` grid (`hparam_search_mean/lr_search.py`), the MLP by a hyperparameter search (`hparam_search_mean/qwen3_4b_2048/`). Both were chosen by 5-fold cross-validation on train (random folds; see the note at the end), so nothing here was chosen by looking at the test set. Same vectors and 90/10 split as above.

## Logistic regression, tuned

Raw inputs and `C = 0.001`: the best of a `C` grid (1e-4 to 1e3) with and without standardizing, at 81.49 CV accuracy (vs. 81.4 for the settings above).

In [7]:
# The cells above standardized X_tr in place; LR wants the raw vectors, so split them again (same split)
X_tr_raw, _, _, _ = train_test_split(X_train, y_train, test_size=0.1, stratify=y_train, random_state=423)
with threadpool_limits(1):
    logreg_tuned = LogisticRegression(C=0.001, max_iter=5000).fit(X_tr_raw, y_tr)
lr_tuned_acc = metrics.accuracy_score(y_test, logreg_tuned.predict(X_test))
print(f"LR test accuracy: {lr_tuned_acc*100:.2f}")

LR test accuracy: 77.46


## MLP, tuned

2560 → 4096 → 64 → 20: the config with the best 3-seed CV accuracy in the search (82.43, vs. 82.27 for RoBERTa `<s>`'s settings above).
- **Dropout:** 0.3 on the input, 0.2 after hidden layer 1, 0.7 after hidden layer 2.
- **AdamW** at lr 1e-5, weight decay 0.1; **label smoothing** 0.2.
- GELU, batch size 32, standardized inputs (the same `X_tr` / `X_dev` as above), early stopping on dev loss with patience 10.

In [8]:
torch.manual_seed(423)

mlp_tuned = nn.Sequential(
    nn.Dropout(0.3),                                    # dropout on the input embedding
    nn.Linear(2560, 4096), nn.GELU(), nn.Dropout(0.2),  # hidden layer 1
    nn.Linear(4096, 64), nn.GELU(), nn.Dropout(0.7),    # hidden layer 2
    nn.Linear(64, n_classes),                           # one logit per class
).to(device)

criterion = nn.CrossEntropyLoss(label_smoothing=0.2)
optimizer = torch.optim.AdamW(mlp_tuned.parameters(), lr=1e-5, weight_decay=0.1)

Same training loop and early stopping as above.

In [9]:
best = {"dev_loss": float("inf"), "epoch": 0, "state": None}
start = time.perf_counter()

for epoch in range(1, 301):
    mlp_tuned.train()  # dropout on
    train_loss = 0
    for xb, yb in loader:  # the same DataLoader over the standardized 90% as above
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        loss = criterion(mlp_tuned(xb), yb)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * len(xb)

    mlp_tuned.eval()  # dropout off
    with torch.no_grad():
        dev_logits = mlp_tuned(X_dev_t)
        dev_loss = nn.functional.cross_entropy(dev_logits, y_dev_t).item()  # no label smoothing, as in the search
        dev_acc = (dev_logits.argmax(dim=1) == y_dev_t).float().mean().item()
    print(f"epoch {epoch:3d}  train loss {train_loss / len(X_tr):.3f}  dev loss {dev_loss:.3f}  dev acc {dev_acc*100:.2f}")

    if dev_loss < best["dev_loss"]:
        best = {"dev_loss": dev_loss, "epoch": epoch, "state": copy.deepcopy(mlp_tuned.state_dict())}
    elif epoch - best["epoch"] >= PATIENCE:
        break

mlp_tuned.load_state_dict(best["state"])
print(f"Training took {time.perf_counter() - start:.1f} s. Using epoch {best['epoch']} (lowest dev loss)")

epoch   1  train loss 2.430  dev loss 1.477  dev acc 73.69


epoch   2  train loss 1.995  dev loss 1.188  dev acc 77.21


epoch   3  train loss 1.923  dev loss 1.091  dev acc 78.74


epoch   4  train loss 1.885  dev loss 1.039  dev acc 79.64


epoch   5  train loss 1.853  dev loss 1.007  dev acc 80.09


epoch   6  train loss 1.840  dev loss 0.979  dev acc 80.18


... (82 outputs left out)

epoch  89  train loss 1.529  dev loss 0.750  dev acc 82.61


epoch  90  train loss 1.531  dev loss 0.752  dev acc 82.61


epoch  91  train loss 1.526  dev loss 0.754  dev acc 82.25


epoch  92  train loss 1.523  dev loss 0.749  dev acc 82.97
Training took 256.5 s. Using epoch 82 (lowest dev loss)


## RoBERTa's settings vs. tuned, on the test set

In [10]:
mlp_tuned.eval()
with torch.no_grad():
    mlp_tuned_pred = mlp_tuned(torch.tensor(scaler.transform(X_test)).to(device)).argmax(dim=1).cpu().numpy()
mlp_tuned_acc = metrics.accuracy_score(y_test, mlp_tuned_pred)

rows = [("RoBERTa <s>'s settings (above)", lr_acc, mlp_acc), ("tuned settings", lr_tuned_acc, mlp_tuned_acc)]
print(f"{'test accuracy':<34}{'LR':>7}{'MLP':>8}")
for name, lr_a, mlp_a in rows:
    print(f"{name:<34}{lr_a*100:>7.2f}{mlp_a*100:>8.2f}")
print(f"(1 point = {len(y_test) / 100:.0f} test messages)")

test accuracy                          LR     MLP
RoBERTa <s>'s settings (above)      77.58   77.71
tuned settings                      77.46   78.06
(1 point = 74 test messages)


The search's own final test retrained this MLP config with 10 seeds on the same split: **78.07 ± 0.19**, and averaging the 10 seeds' predicted probabilities (an ensemble) gives **78.11** (`hparam_search_mean/final_test_qwen3_4b_2048.json`). Read the single seed above against those.

Everything was tuned by CV on train; the test set was only used to score the final settings (here and in the final test).

**Two later findings** (`hparam_search_mean/Qwen4B2048Search.md`):
- **The random CV folds used here leak threads.** 65% of dev posts have another post from their thread (same subject line) in the fold's training part, but only 32% of test posts have one in train. That's why CV (about 82) is so far above test (about 78). Folds grouped by thread score about 3 points lower.
- **The instruction mattered more than the window.** Re-embedding with "Classify the given Usenet newsgroup post by the newsgroup it was posted to" instead of Qwen's MTEB prompt gained about 1 point of thread-grouped CV, vs. about 0.1 for 2,048 tokens instead of 512.